In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("my-app") \
    .master("local[*]") \
    .getOrCreate()

In [2]:
# create a DataFrame with numbers from 0 to 999
my_range = spark.range(1000).toDF("number")

In [3]:
even_numbers = my_range.where("number % 2 = 0")

In [4]:
even_numbers.count()

500

In [13]:
flight_data_2015 = spark\
    .read \
    .option("inferSchema", "true") \
    .option("header", "true") \
    .csv("../data/flight-data/2015-summary.csv")

In [14]:
flight_data_2015.take(3)

[Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Romania', count=15),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Croatia', count=1),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Ireland', count=344)]

In [15]:
flight_data_2015.sort("count").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [count#52 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(count#52 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=90]
      +- FileScan csv [DEST_COUNTRY_NAME#50,ORIGIN_COUNTRY_NAME#51,count#52] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/data/flight-data/2015-summary.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string,ORIGIN_COUNTRY_NAME:string,count:int>




In [23]:
spark.conf.set("spark.sql.shuffle.partitions", "15")

In [24]:
flight_data_2015.sort("count").take(2)

[Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Singapore', count=1),
 Row(DEST_COUNTRY_NAME='Moldova', ORIGIN_COUNTRY_NAME='United States', count=1)]

In [26]:
# make a DataFrame into a table or view
flight_data_2015.createOrReplaceTempView("flight_data_2015")

In [27]:
sql_way = spark.sql("""
SELECT dest_country_name, count(1)
FROM flight_data_2015
GROUP BY dest_country_name
""")

In [28]:
data_frame_way = flight_data_2015 \
    .groupBy("DEST_COUNTRY_NAME") \
    .count()

In [29]:
sql_way.explain()
data_frame_way.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[dest_country_name#50], functions=[count(1)])
   +- Exchange hashpartitioning(dest_country_name#50, 15), ENSURE_REQUIREMENTS, [plan_id=130]
      +- HashAggregate(keys=[dest_country_name#50], functions=[partial_count(1)])
         +- FileScan csv [DEST_COUNTRY_NAME#50] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/data/flight-data/2015-summary.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string>


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[DEST_COUNTRY_NAME#50], functions=[count(1)])
   +- Exchange hashpartitioning(DEST_COUNTRY_NAME#50, 15), ENSURE_REQUIREMENTS, [plan_id=143]
      +- HashAggregate(keys=[DEST_COUNTRY_NAME#50], functions=[partial_count(1)])
         +- FileScan csv [DEST_COUNTRY_NAME#50] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 pat

In [30]:
spark.sql("SELECT max(count) FROM flight_data_2015").take(1)

[Row(max(count)=370002)]

In [32]:
from pyspark.sql.functions import max

flight_data_2015.select(max("count")).take(1)

[Row(max(count)=370002)]

In [34]:
max_sql = spark.sql("""
SELECT dest_country_name, sum(count) as destination_total
FROM flight_data_2015
GROUP BY dest_country_name
ORDER BY sum(count) DESC
LIMIT 5
""")

max_sql.show()

+-----------------+-----------------+
|dest_country_name|destination_total|
+-----------------+-----------------+
|    United States|           411352|
|           Canada|             8399|
|           Mexico|             7140|
|   United Kingdom|             2025|
|            Japan|             1548|
+-----------------+-----------------+



In [36]:
from pyspark.sql.functions import desc

flight_data_2015 \
    .groupBy("dest_country_name") \
    .sum("count") \
    .withColumnRenamed("sum(count)", "destination_total") \
    .sort(desc("destination_total")) \
    .limit(5) \
    .show()

+-----------------+-----------------+
|dest_country_name|destination_total|
+-----------------+-----------------+
|    United States|           411352|
|           Canada|             8399|
|           Mexico|             7140|
|   United Kingdom|             2025|
|            Japan|             1548|
+-----------------+-----------------+



In [37]:
flight_data_2015 \
    .groupBy("dest_country_name") \
    .sum("count") \
    .withColumnRenamed("sum(count)", "destination_total") \
    .sort(desc("destination_total")) \
    .limit(5) \
    .explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- TakeOrderedAndProject(limit=5, orderBy=[destination_total#160L DESC NULLS LAST], output=[dest_country_name#50,destination_total#160L])
   +- HashAggregate(keys=[dest_country_name#50], functions=[sum(count#52)])
      +- Exchange hashpartitioning(dest_country_name#50, 15), ENSURE_REQUIREMENTS, [plan_id=313]
         +- HashAggregate(keys=[dest_country_name#50], functions=[partial_sum(count#52)])
            +- FileScan csv [DEST_COUNTRY_NAME#50,count#52] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/data/flight-data/2015-summary.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<DEST_COUNTRY_NAME:string,count:int>


